
# Notebook de ingestão da tabela online_retail

In [0]:
from pathlib import Path
import sys

# Identifica a raiz do repositório
current_path = Path.cwd()

project_root = (
    current_path.parent
    if current_path.name == "notebooks"
    else current_path
)

# Adiciona src ao caminho de imports
src_path = project_root / "src"

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Validação
print(f"Raiz do projeto: {project_root}")
print(f"Caminho src: {src_path}")

print(
    "data_quality.py encontrado:",
    (
        src_path
        / "reactivation_model"
        / "data_quality.py"
    ).exists()
)

In [0]:
from pyspark.sql.functions import *
from reactivation_model.data_quality import run_data_quality_checks


## Leitura das tabelas

In [0]:
df = spark.sql(
    '''
    select * from workspace.bronze_layer.online_retail_i
    union all 
    select * from workspace.bronze_layer.online_retail_ii
    '''
)


## Testes de qualidade

Ao carregar o notebook já sabemos que temos alguns erros em: 

```
❌ Valores nulos encontrados:
   Description: 4,382
   Customer ID: 243,007

❌ 34,335 linhas duplicadas encontradas

======================================================================
❌ DATA QUALITY REPROVADO
Falhas encontradas: ['Valores nulos encontrados', 'Linhas duplicadas encontradas']
```

Devido a isso será corrigido esses tópicos acima:

- Exclusão dos customer_id's nulos
- Dedup do dataset


## Tratamento do dataset

In [0]:
df_silver = (
    df
    .filter(col('Customer ID').isNotNull()) # Exclusão de Customer_id's nulos
    .dropDuplicates() # Excluindo duplicatas
    .withColumnRenamed(
        "Customer ID",
        "customer_id"
    )
    .withColumnRenamed(
        "Invoice",
        "invoice"
    )
    .withColumnRenamed(
        "StockCode",
        "stock_code"
    )
    .withColumnRenamed(
        "Description",
        "description"
    )
    .withColumnRenamed(
        "Quantity",
        "quantity"
    )
    .withColumnRenamed(
        "InvoiceDate",
        "invoice_date"
    )
    .withColumnRenamed(
        "Price",
        "price"
    )
    .withColumnRenamed(
        "Country",
        "country"
    )
)

In [0]:
run_data_quality_checks(
    df=df_silver,
    date_column="invoice_date"
)


## Salvando a tabela Silver

In [0]:
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS workspace.silver_layer
""")

In [0]:
(
    df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "silver_layer.online_retail_transactions"
    )
)